## Introduction to data ingestion

In [2]:
import os
from typing import List, Dict, Any
import pandas as pd
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)
print("All imports successful!")

/Users/namitkumar/programming/rag/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful!


## Understand the document structure in langchain

In [3]:
# creating a sample document

doc = Document(
    page_content="This is a sample document for testing the text splitters.",
    metadata={"source": "test_document.txt"}
)

print("Sample document created:")
print(f"Content: {doc.page_content}")
print(f"Metadata: {doc.metadata}")

Sample document created:
Content: This is a sample document for testing the text splitters.
Metadata: {'source': 'test_document.txt'}


### Reading a Textfile (.txt)

In [4]:
# creating a simple textfile
import os 
os.makedirs("data/textfiles", exist_ok=True)

In [5]:
sample_text = {
    "data/textfiles/python_intro.txt" : """Python is a high-level, interpreted programming language known for its simplicity and readability. 
      It was created by Guido van Rossum and first released in 1991. 
      Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming. 
      It has a large standard library that provides tools suited to many tasks, making it a popular choice for web development, data analysis, artificial intelligence, scientific computing, and more. 
      Python's syntax emphasizes code readability, allowing developers to express concepts in fewer lines of code compared to other languages.""",
    "data/textfiles/python_basics.txt" : """Python basics include variables, data types, control structures, and functions. 
      Variables are used to store data, and Python has several built-in data types such as integers, floats, strings, and booleans. 
      Control structures like if-else statements and loops allow you to control the flow of your program. 
      Functions are reusable blocks of code that perform specific tasks."""

}

for file_path, content in sample_text.items():
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created successfully!")

Sample text files created successfully!


### Textloader Reading the single file


In [6]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/textfiles/python_intro.txt", encoding="utf-8")

documents = loader.load()
print(type(documents))
print(documents)
print(documents[0].metadata)


<class 'list'>
[Document(metadata={'source': 'data/textfiles/python_intro.txt'}, page_content="Python is a high-level, interpreted programming language known for its simplicity and readability. \n      It was created by Guido van Rossum and first released in 1991. \n      Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming. \n      It has a large standard library that provides tools suited to many tasks, making it a popular choice for web development, data analysis, artificial intelligence, scientific computing, and more. \n      Python's syntax emphasizes code readability, allowing developers to express concepts in fewer lines of code compared to other languages.")]
{'source': 'data/textfiles/python_intro.txt'}


## DirectoryLoader

In [7]:
from langchain_community.document_loaders import DirectoryLoader

directory_loader = DirectoryLoader("data/textfiles", 
                                   glob="**/*.txt", 
                                   show_progress=True, 
                                   loader_cls=TextLoader, )

all_documents = directory_loader.load()
print(all_documents)

print(f"Total documents loaded: {len(all_documents)}")
for i, doc in enumerate(all_documents):
    print(f"Document {i+1}:")
    print(f"Source: {doc.metadata['source']}")

100%|██████████| 2/2 [00:00<00:00, 2652.10it/s]

[Document(metadata={'source': 'data/textfiles/python_intro.txt'}, page_content="Python is a high-level, interpreted programming language known for its simplicity and readability. \n      It was created by Guido van Rossum and first released in 1991. \n      Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming. \n      It has a large standard library that provides tools suited to many tasks, making it a popular choice for web development, data analysis, artificial intelligence, scientific computing, and more. \n      Python's syntax emphasizes code readability, allowing developers to express concepts in fewer lines of code compared to other languages."), Document(metadata={'source': 'data/textfiles/python_basics.txt'}, page_content='Python basics include variables, data types, control structures, and functions. \n      Variables are used to store data, and Python has several built-in data types such as integers, floats, string

### Text Splitting Techniques

In [8]:
# Method 1 : Using Character Text Splitter
text = all_documents[0].page_content


print("Using CharacterTextSplitter:")
char_splitter = CharacterTextSplitter(separator = "\n",
                                      chunk_size=10, 
                                      chunk_overlap=2,
                                      length_function=len,
                                      )

char_chunks = char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks:")
print(f"Chunks: {char_chunks[0][:100]}...")


Created a chunk of size 99, which is longer than the specified 10
Created a chunk of size 69, which is longer than the specified 10
Created a chunk of size 121, which is longer than the specified 10
Created a chunk of size 200, which is longer than the specified 10


Using CharacterTextSplitter:
Created 5 chunks:
Chunks: Python is a high-level, interpreted programming language known for its simplicity and readability....


In [10]:
print(char_chunks[0])
print(char_chunks[1])

Python is a high-level, interpreted programming language known for its simplicity and readability.
It was created by Guido van Rossum and first released in 1991.


In [14]:
# Method 2 : Using Recursive Character Text Splitter

recursive_splitter = RecursiveCharacterTextSplitter(
    separators=[" "],
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

recursive_chunks = recursive_splitter.split_text(text)
print(f"Created {len(recursive_chunks)} chunks:")
print(f"Chunks: {recursive_chunks[0][:100]}...")

Created 4 chunks:
Chunks: Python is a high-level, interpreted programming language known for its simplicity and readability. 
...


In [15]:
print(recursive_chunks[0])
print("--------")
print(recursive_chunks[1])

Python is a high-level, interpreted programming language known for its simplicity and readability. 
      It was created by Guido van Rossum and first released in 1991. 
      Python supports multiple
--------
supports multiple programming paradigms, including procedural, object-oriented, and functional programming. 
      It has a large standard library that provides tools suited to many tasks, making it
